# Ozon E-CUP 2026 — CE tiny 2-epoch stage-A only

Отличия от ce-v1: EPOCHS_A=2 (две эпохи LLM-претрейна), стадия B отключена
(A/B-тест на LB: stage-A 0.450 > A+B 0.4355). Всё остальное идентично.

Ориентиры: ce-v1 stage-A после 1 эпохи llm-val-fast 0.746 → LB 0.450.
Если к концу 2-й эпохи llm-val-fast > 0.75 — сабмитим.


In [1]:
import os, json, re, gc, time, math
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

BASE = "/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items"  # <-- поправь
MATCHES_PATH = f"{BASE}/matches.parquet"
MATCHES_LLM_PATH = f"{BASE}/matches_llm.parquet"
ITEMS_PATH = f"{BASE}/items.parquet"          # полный, 4.1GB
ITEMS_HUMAN_PATH = f"{BASE}/items_human.parquet"

MODEL_NAME = "cointegrated/rubert-tiny2"   # 29M параметров, MIT, русский
MAX_LEN = 160
BATCH = 256
LR_A, LR_B = 2e-4, 5e-5     # стадия A: LLM-пары; стадия B: ручные
EPOCHS_A, EPOCHS_B = 2, 0
WARMUP = 2000
EVAL_EVERY = 10000          # шагов
CKPT = "/kaggle/working/ce_tiny2ep_ckpt.pt"
OUT_DIR = "/kaggle/working/ce_tiny2ep_final"
SEED = 42

torch.manual_seed(SEED); np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device, torch.cuda.get_device_name(0) if device == "cuda" else "")

cuda Tesla T4


## 1. Текст товара
Имя + компактные атрибуты (коды, бренд, вариантные). Эту же функцию потом копируем в сабмит — менять только синхронно!

In [2]:
KEY_ORDER = ["бренд", "артикул", "партномер", "oem", "код", "модель", "размер",
             "цвет", "объем", "обьем", "вес", "тип", "материал", "количество"]

def build_text(name, attributes, max_attr_chars=260):
    parts = [str(name) if name is not None else ""]
    try:
        attrs = json.loads(attributes) if isinstance(attributes, str) else {}
    except Exception:
        attrs = {}
    if isinstance(attrs, dict) and attrs:
        low = {str(k).lower(): str(v) for k, v in attrs.items() if v}
        picked, used = [], set()
        for want in KEY_ORDER:
            for k, v in low.items():
                if want in k and k not in used:
                    picked.append(f"{k}:{v}"); used.add(k)
        rest = [f"{k}:{v}" for k, v in low.items() if k not in used]
        s = " ; ".join(picked + rest)[:max_attr_chars]
        parts.append(s)
    return " | ".join(parts)

t0 = time.time()
item_text, item_cat = {}, {}
f = pq.ParquetFile(ITEMS_PATH)
for b in f.iter_batches(columns=["id", "name", "attributes", "category"], batch_size=500_000):
    for i, n, a, c in b.to_pandas().itertuples(index=False, name=None):
        item_text[i] = build_text(n, a)
        item_cat[i] = c
print(f"items: {len(item_text):,} за {time.time()-t0:.0f}s")

items: 13,397,761 за 572s


## 2. Сплиты — ИДЕНТИЧНЫ нашим GBM-экспериментам
Групповые (union-find по товарам): manual seed 42 / 20%, LLM seed 13 / 3% компонент.

In [3]:
def group_val_mask(df, val_frac, seed):
    parent = {}
    def find(x):
        p = parent.setdefault(x, x)
        while p != parent[p]:
            parent[p] = parent[parent[p]]; p = parent[p]
        parent[x] = p; return p
    for a, b in zip(df.id1.values, df.id2.values):
        ra, rb = find(a), find(b)
        if ra != rb: parent[rb] = ra
    comp = np.fromiter((find(i) for i in df.id1.values), dtype=np.int64, count=len(df))
    rng = np.random.RandomState(seed)
    uniq = np.unique(comp)
    val_set = set(uniq[rng.rand(len(uniq)) < val_frac].tolist())
    return np.fromiter((c in val_set for c in comp), dtype=bool, count=len(df))

m = pd.read_parquet(MATCHES_PATH)
m_val_mask = group_val_mask(m, 0.20, 42)
m_train, m_val = m[~m_val_mask].copy(), m[m_val_mask].copy()

ml = pd.read_parquet(MATCHES_LLM_PATH)
l_val_mask = group_val_mask(ml, 0.03, 13)
ml_train, ml_val = ml[~l_val_mask].copy(), ml[l_val_mask].copy()
ml_val = ml_val[(ml_val.target <= 0.2) | (ml_val.target >= 0.8)].copy()
ml_val["target"] = (ml_val.target >= 0.5).astype(int)

for df in (m_train, m_val, ml_train, ml_val):
    df["category"] = [item_cat[i] for i in df.id1]
del m, ml, m_val_mask, l_val_mask
import gc; gc.collect()
print(f"manual: {len(m_train):,}/{len(m_val):,}  llm: {len(ml_train):,}/{len(ml_val):,}")

manual: 292,706/72,948  llm: 10,950,394/191,555


## 3. Dataset / метрика

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PairDS(Dataset):
    def __init__(self, pairs_df):
        self.id1 = pairs_df.id1.values
        self.id2 = pairs_df.id2.values
        self.y = pairs_df.target.values.astype(np.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return item_text[self.id1[i]], item_text[self.id2[i]], self.y[i]

def collate(batch):
    t1, t2, y = zip(*batch)
    enc = tokenizer(list(t1), list(t2), padding=True, truncation=True,
                    max_length=MAX_LEN, return_tensors="pt")
    return enc, torch.tensor(y)

@torch.no_grad()
def predict(model, pairs_df, bs=512):
    model.eval()
    dl = DataLoader(PairDS(pairs_df), batch_size=bs, collate_fn=collate,
                    num_workers=0, shuffle=False)
    out = []
    for enc, _ in dl:
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(**enc).logits.squeeze(-1)
        out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(out)

def macro_pr_auc(pairs_df, preds):
    z = pairs_df[["category", "target"]].copy(); z["pred"] = preds
    aps = z.groupby("category").apply(lambda g: average_precision_score(g.target, g.pred))
    return float(aps.mean()), aps

# быстрый eval-сэмпл, чтобы не гонять весь val каждые 20k шагов
ml_val_fast = ml_val.sample(min(60_000, len(ml_val)), random_state=0)

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## 4. Стадия A — обучение на LLM-парах (soft labels)

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).to(device)

def train_stage(pairs_df, epochs, lr, tag):
    dl = DataLoader(PairDS(pairs_df), batch_size=BATCH, collate_fn=collate,
                    num_workers=0, shuffle=True, drop_last=True)
    steps_total = len(dl) * epochs
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, min(WARMUP, steps_total // 10), steps_total)
    scaler = torch.amp.GradScaler()
    lossf = nn.BCEWithLogitsLoss()
    step, t0, run_loss = 0, time.time(), 0.0
    best = 0.0
    for ep in range(epochs):
        for enc, y in dl:
            model.train()
            enc = {k: v.to(device, non_blocking=True) for k, v in enc.items()}
            y = y.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(**enc).logits.squeeze(-1)
                loss = lossf(logits, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            run_loss += loss.item(); step += 1
            if step % 2000 == 0:
                sps = step * BATCH / (time.time() - t0)
                print(f"[{tag}] ep{ep} step {step}/{steps_total} loss={run_loss/2000:.4f} {sps:.0f} pair/s", flush=True)
                run_loss = 0.0
            if step % EVAL_EVERY == 0:
                mac, _ = macro_pr_auc(ml_val_fast, predict(model, ml_val_fast))
                print(f"[{tag}] step {step}: llm-val-fast Macro PR-AUC = {mac:.4f}", flush=True)
                torch.save({"model": model.state_dict(), "step": step, "metric": mac}, CKPT)
                if mac > best: best = mac
    return best

train_stage(ml_train, EPOCHS_A, LR_A, "A/llm")

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider trai

[A/llm] ep0 step 2000/85548 loss=0.5064 596 pair/s
[A/llm] ep0 step 4000/85548 loss=0.4816 591 pair/s
[A/llm] ep0 step 6000/85548 loss=0.4630 591 pair/s
[A/llm] ep0 step 8000/85548 loss=0.4375 592 pair/s
[A/llm] ep0 step 10000/85548 loss=0.4216 593 pair/s
[A/llm] step 10000: llm-val-fast Macro PR-AUC = 0.6322


/tmp/ipykernel_23/3698110657.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  aps = z.groupby("category").apply(lambda g: average_precision_score(g.target, g.pred))


...

/tmp/ipykernel_23/3698110657.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  aps = z.groupby("category").apply(lambda g: average_precision_score(g.target, g.pred))


[A/llm] ep1 step 72000/85548 loss=0.3354 585 pair/s
[A/llm] ep1 step 74000/85548 loss=0.3348 585 pair/s
[A/llm] ep1 step 76000/85548 loss=0.3332 585 pair/s
[A/llm] ep1 step 78000/85548 loss=0.3331 585 pair/s
[A/llm] ep1 step 80000/85548 loss=0.3328 586 pair/s
[A/llm] step 80000: llm-val-fast Macro PR-AUC = 0.7737


/tmp/ipykernel_23/3698110657.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  aps = z.groupby("category").apply(lambda g: average_precision_score(g.target, g.pred))


[A/llm] ep1 step 82000/85548 loss=0.3310 585 pair/s
[A/llm] ep1 step 84000/85548 loss=0.3313 585 pair/s


0.7736583641126911

## 5. Стадия B — дообучение на ручных парах

In [6]:
print("stage B отключена по итогам A/B-теста: stage-A only лучше на LB (0.450 vs 0.4355)")

stage B отключена по итогам A/B-теста: stage-A only лучше на LB (0.450 vs 0.4355)


## 6. Финальные метрики — сравнивать с GBM: llm 0.608 / manual 0.625

In [7]:
mac_l, aps_l = macro_pr_auc(ml_val, predict(model, ml_val))
print(f"[llm holdout, уверенные] Macro PR-AUC = {mac_l:.4f}")
print(aps_l.round(3).to_string())

mac_m, aps_m = macro_pr_auc(m_val, predict(model, m_val))
print(f"\n[manual holdout] Macro PR-AUC = {mac_m:.4f}")
print(aps_m.round(3).to_string())

/tmp/ipykernel_23/3698110657.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  aps = z.groupby("category").apply(lambda g: average_precision_score(g.target, g.pred))


[llm holdout, уверенные] Macro PR-AUC = 0.7714
category
Автотовары                 0.936
Аптека                     0.898
Бытовая техника            0.900
Бытовая химия              0.882
Галантерея и аксессуары    0.552
Детские товары             0.811
Дом и сад                  0.793
Канцелярские товары        0.801
Красота и гигиена          0.768
Мебель                     0.808
Музыкальные инструменты    0.909
Обувь                      0.324
Одежда                     0.395
Продукты питания           0.818
Спорт и отдых              0.811
Строительство и ремонт     0.855
Товары для животных        0.886
Хобби и творчество         0.877
Электроника                0.801
Ювелирные изделия          0.604

[manual holdout] Macro PR-AUC = 0.6492
category
Автотовары                 0.626
Аптека                     0.695
Бытовая техника            0.762
Бытовая химия              0.837
Галантерея и аксессуары    0.513
Детские товары             0.834
Дом и сад                  0.704
Канц

/tmp/ipykernel_23/3698110657.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  aps = z.groupby("category").apply(lambda g: average_precision_score(g.target, g.pred))


## 7. Сохранение — скачать папку целиком и прислать для сборки сабмита

In [8]:
os.makedirs(OUT_DIR, exist_ok=True)
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
with open(f"{OUT_DIR}/metrics.json", "w") as fp:
    json.dump({"llm_holdout_macro_prauc": mac_l, "manual_holdout_macro_prauc": mac_m,
               "model": MODEL_NAME, "max_len": MAX_LEN,
               "llm_by_cat": aps_l.to_dict(), "manual_by_cat": aps_m.to_dict()}, fp,
              ensure_ascii=False, indent=1)
print("saved:", OUT_DIR)
!cd /kaggle/working && zip -qr ce_tiny2ep_final.zip ce_tiny2ep_final && du -h ce_tiny2ep_final.zip

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved: /kaggle/working/ce_tiny2ep_final
104M	ce_tiny2ep_final.zip
